# TweetAge v7: modelo de lenguaje neuronal de caracteres por clase

**Objetivo.** Ultimo intento dentro de la familia generativa: sustituir el componente de caracteres Kneser-Ney por un modelo de lenguaje recurrente de caracteres entrenado desde cero por el equipo, uno por rango de edad, y clasificar por verosimilitud como en v6. Despues de esta corrida se cierra la experimentacion y se escribe el informe.

**Por que este modelo.** Las rondas previas muestran un techo de la familia de n-gramas con Kneser-Ney cercano a 0,44 de F1 macro OOF:

- v5 (Kneser-Ney, particion con fuga de 2.561 textos): holdout 0,44945 inflado; Kaggle publico 0,45643.
- v6 (Kneser-Ney modificado, particion agrupada): OOF sin calibrar 0,44106; OOF calibrado 0,44353; holdout 0,41847; Kaggle publico 0,45737. Version valida publicada.
- v7 ordinal (cortes por clase): OOF 0,42321; NO PASA.
- v7 descuento con concentracion: OOF 0,440875 (theta=0,5) y 0,440788 (theta=1,0); NO PASA.
- v7 normalizacion (particion v7_lower en minusculas): A 0,440081; B 0,439709; C 0,436893; NO PASA.
- v7 hiperparametros (8 configuraciones sobre v7_lower): ninguna supera la referencia recalculada 0,440081; la mejor, C7 (car 5, pal 4, lambda 0,6), con diferencia media -0,002367; NO PASA. Familia de n-gramas cerrada.

Suavizado, normalizacion e hiperparametros movieron el resultado unas milesimas. Un modelo recurrente puede capturar dependencias de caracteres mas largas que el orden 4-5 de los n-gramas.

**Inspiracion.** Peng et al. (2003), modelos de caracteres por clase para atribucion de autoria; Teahan y Harper (2003), clasificacion por modelos de lenguaje comparados por verosimilitud; Chen y Goodman (1999), base de Kneser-Ney, que se mantiene como componente de palabras; Mikolov et al. (2010), modelo de lenguaje recurrente entrenado desde cero; Sutskever, Martens y Hinton (2011), modelo recurrente a nivel de caracteres; Jozefowicz et al. (2016), buenas practicas de entrenamiento y control de sobreajuste. Bengio et al. (2003) solo como antecedente: aqui no se usan embeddings.

**Interpretacion.** Es un modelo de lenguaje propio, generativo y por clase: cada GRU estima P(caracter siguiente | historia) con los textos de una clase y el tweet se asigna por log-verosimilitud mas log prior. No es un embedding ni una representacion vectorial reutilizada: la entrada es one-hot de caracteres, no hay tabla de vectores de entrada ni pesos preentrenados, y todos los parametros se aprenden desde cero dentro de cada pliegue.

**Modelo.** Vocabulario: caracteres del ajuste del pliegue mas <bos>, <eos> y <unk>. Entrada one-hot, una capa GRU de 128 unidades, dropout 0,2 sobre la salida recurrente, softmax sobre el vocabulario. Adam (2e-3), lote 64, longitud maxima 280, maximo 8 epocas, parada temprana (paciencia 2) sobre una validacion interna formada por el 10 % de los grupos del propio ajuste del pliegue, nunca sobre el pliegue de evaluacion. Puntuacion: log-verosimilitud total del texto mas <eos>. Combinacion: score = 0,75 x neuronal_caracteres + 0,25 x KN_palabras(orden 3, como v6) + log prior. Sin calibracion de sesgos. Normalizacion A de v6.

**Referencia A.** Combinacion de v6 sobre v7_lower: 0,75 x KN_caracteres(4) + 0,25 x KN_palabras(3) + log prior, recalculada en esta corrida en los mismos pliegues.

**Regla de decision (fijada antes de ejecutar).** Etapa 1: solo pliegue 1; se continua si F1(pliegue 1) - 0,440843 > 0; si no, "NO PASA EN ETAPA 1". Etapa 2: se acepta solo si la media de las cinco diferencias pareadas de F1 macro contra la referencia A recalculada es > 0 y mayor que su desviacion estandar muestral (ddof=1). Heuristica, no prueba formal. Si no pasa, se omiten holdout y CSV. El modelo neuronal solo (sin KN de palabras) es descriptivo y no decide. Si un pliegue supera 60 minutos, se detiene la corrida.


In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import subprocess
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(6)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
CHAR_ORDERS=[4]
WORD_ORDERS=[3]
REF_CHAR_ORDER,KN_WORD_ORDER,REF_WEIGHT=4,3,0.75
NEURAL_WEIGHT=0.75
HIPERPARAMETROS_PASO=False
NEURAL=dict(hidden=128,dropout=0.2,lr=2e-3,batch=64,max_len=280,max_epochs=8,patience=2,internal_val_fraction=0.1,grad_clip=5.0,score_batch=512)
FOLD_LIMIT_SECONDS=3600
REF_FOLD1_MEASURED=0.440843
roots=[Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR=next((candidate for candidate in roots if (candidate/'TweetAge_v7_lm_neuronal.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH=OUT_DIR/'TweetAge_v7_lm_neuronal.ipynb'
SUB_PATH=OUT_DIR/'age_submission_v7_lm_neuronal.csv'
MODEL_DIR=OUT_DIR/'modelos_v7_neuronal'
SPLIT_PATH=OUT_DIR/'split_idx_v7_lower_seed42.npz'
DATA_DIR=OUT_DIR/'tweet-age-2026-puj'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
assert not MODEL_DIR.exists(),'La carpeta de modelos ya existe; no se sobrescribe.'
INPUT_HASHES={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in OUT_DIR.iterdir() if path.is_file() and path!=NOTEBOOK_PATH}
assert PROTECTED_HASHES[SPLIT_PATH.name]=='692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83','Hash del NPZ v7_lower distinto; detener.'
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__,'torch',torch.__version__,'hilos',torch.get_num_threads(),'dispositivo CPU')
print('Semilla',SEED,'(numpy y torch); algoritmos deterministas de torch activados.')
print('SHA-256 NPZ v7_lower:',PROTECTED_HASHES[SPLIT_PATH.name])
print('SHA-256 age_train.csv:',INPUT_HASHES['age_train.csv'])
print('KN de palabras: orden',KN_WORD_ORDER,'(configuracion de referencia de v6), porque v7 hiperparametros NO PASO (HIPERPARAMETROS_PASO =',HIPERPARAMETROS_PASO,').')
print('Configuracion neuronal:',NEURAL)

def check_compliance(notebook):
    # Guardia AST de v6 mas prohibiciones de esta ronda: vectorizadores, embeddings, tablas de vectores y pesos preentrenados.
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'embeddingbag', 'word2vec', 'glove', 'gensim', 'from_pretrained', 'pretrained', 'load_state_dict', 'hub', 'load_url', 'model_zoo', 'torchtext', 'keyedvectors'}
    problems = []
    uses_one_hot = False
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
                if node.attr == 'one_hot':
                    uses_one_hot = True
                if node.attr == 'load' and not (isinstance(node.value, ast.Name) and node.value.id in {'np', 'json'}):
                    problems.append((cell_number, node.lineno, 'carga de pesos'))
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    if not uses_one_hot:
        raise RuntimeError('Guardia AST: la entrada debe ser one-hot de caracteres.')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar; sin vectorizadores, embeddings, tablas de vectores ni pesos preentrenados; entrada one-hot.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1 torch 2.14.1 hilos 6 dispositivo CPU
Semilla 42 (numpy y torch); algoritmos deterministas de torch activados.
SHA-256 NPZ v7_lower: 692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83
SHA-256 age_train.csv: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
KN de palabras: orden 3 (configuracion de referencia de v6), porque v7 hiperparametros NO PASO (HIPERPARAMETROS_PASO = False ).
Configuracion neuronal: {'hidden': 128, 'dropout': 0.2, 'lr': 0.002, 'batch': 64, 'max_len': 280, 'max_epochs': 8, 'patience': 2, 'internal_val_fraction': 0.1, 'grad_clip': 5.0, 'score_batch': 512}
Guardia AST de v6 superada antes de entrenar; sin vectorizadores, embeddings, tablas de vectores ni pesos preentrenados; entrada one-hot.


## Datos, normalizacion A y verificacion del NPZ v7_lower
Celda identica a la de v7 hiperparametros: hashes del train y de los grupos, cero fuga en minusculas en holdout y en los cinco pliegues.

In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns=verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw=pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean.loc[~excluded]
train=train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
train['normalized']=train.text.map(normalize_text)
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:
    split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==INPUT_HASHES['age_train.csv'] and int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_keys=np.array([text.lower() for text in train.normalized],dtype=object)
groups_hash=hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert groups_hash==str(split['groups_sha256'])
group_ids,unique_keys=pd.factorize(group_keys,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert set(group_ids[idx_development])==set(dev_groups) and set(group_ids[idx_holdout])==set(holdout_groups)
assert not np.intersect1d(dev_groups,holdout_groups).size
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
group_counts=np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
texts_development=train.normalized.iloc[idx_development].to_numpy()
y_development=y_all[idx_development]
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[dict(validacion='holdout',textos_compartidos=0)]
for fold_number,(fit_positions,app_positions) in enumerate(group_folds,1):
    fit_idx=np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx=np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    shared={text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    leak_rows.append(dict(validacion=str(fold_number),textos_compartidos=len(shared)))
    if shared:
        print('DETENER: fuga en pliegue',fold_number,'ejemplos',sorted(shared)[:10])
        raise RuntimeError('Fuga en minusculas; no entrenar.')
    folds.append((fit_idx,app_idx))
print('NPZ REUTILIZADO SIN REGENERAR:',SPLIT_PATH.name)
print('SHA-256 train y grupos correctos. SHA-256 grupos:',groups_hash)
print('Filas train original/limpio/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Textos solo URL:',int(train.normalized.str.strip().eq('<url>').sum()),'solo usuario:',int(train.normalized.str.strip().eq('<user>').sum()))
display(pd.DataFrame(leak_rows))


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ REUTILIZADO SIN REGENERAR: split_idx_v7_lower_seed42.npz
SHA-256 train y grupos correctos. SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Filas train original/limpio/excluidas: 169336 169181 155
Desarrollo grupos/filas: 126309 132987 holdout grupos/filas: 31184 36194
Textos solo URL: 3785 solo usuario: 287


,validacion,textos_compartidos
0,holdout,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


## Kneser-Ney modificado de v6 (referencia A y componente de palabras)

In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def test_shared_counts():
    toy_texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(toy_texts,level)
        sequences=[encode_text(text,level,mapping) for text in toy_texts]
        support=set(mapping.values())
        shared_model=ModifiedKN(max(orders),support).fit(sequences)
        for order in orders:
            separate_model=ModifiedKN(order,support).fit(sequences)
            for sequence in sequences+[UNK]:
                assert abs(shared_model.log_probability(sequence,order)-separate_model.log_probability(sequence,order))<1e-12,(level,order)
            separate_model.close()
        shared_model.close()
    print('Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.')

test_modified_kn()
test_shared_counts()
timing_rows=[]

def fit_score_orders(fit_texts,fit_labels,app_texts,level,orders,phase,fold):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    preprocessing=time.perf_counter()-started
    matrices={order:np.zeros((len(app_texts),6)) for order in orders}
    fit_seconds=0.0
    score_seconds={order:0.0 for order in orders}
    for class_index in range(6):
        fit_started=time.perf_counter()
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        fit_seconds+=time.perf_counter()-fit_started
        for order in orders:
            score_started=time.perf_counter()
            matrices[order][:,class_index]=[model.log_probability(sequence,order) for sequence in encoded_app]
            score_seconds[order]+=time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
    for order in orders:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,orden=order,ajuste_compartido_s=preprocessing+fit_seconds,puntuacion_s=score_seconds[order],tiempo_amortizado_s=score_seconds[order]+(preprocessing+fit_seconds)/len(orders)))
    print(phase,fold,level,'ordenes',orders,'segundos',round(time.perf_counter()-started,2),flush=True)
    return matrices


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.


## Historial de intentos

In [4]:
history=pd.DataFrame([
    dict(version='v5',descripcion='Kneser-Ney; particion con fuga de 2.561 textos',f1_oof='0,43847 (calibrado)',estado='Kaggle 0,45643; holdout 0,44945 inflado'),
    dict(version='v6',descripcion='Kneser-Ney modificado; particion agrupada',f1_oof='0,44106 sin calibrar / 0,44353 calibrado',estado='Valida publicada; Kaggle 0,45737; holdout 0,41847'),
    dict(version='v7 ordinal',descripcion='Cortes por clase, decodificacion ordinal',f1_oof='0,42321',estado='NO PASA'),
    dict(version='v7 descuento',descripcion='Interpolacion descontada con concentracion',f1_oof='0,440875 (theta 0,5) / 0,440788 (theta 1,0)',estado='NO PASA'),
    dict(version='v7 normalizacion',descripcion='Particion v7_lower; variantes A/B/C',f1_oof='0,440081 / 0,439709 / 0,436893',estado='NO PASA'),
    dict(version='v7 hiperparametros',descripcion='8 configuraciones de ordenes y lambda',f1_oof='mejor C7 0,437677 (dif. media -0,002367)',estado='NO PASA'),
    dict(version='v7 neuronal',descripcion='GRU de caracteres por clase + KN palabras',f1_oof='(se llena al final)',estado='(se llena al final)'),
])
print('Historial de intentos. Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.')
display(history)


Historial de intentos. Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.


,version,descripcion,f1_oof,estado
0,v5,Kneser-Ney; particion con fuga de 2.561 textos,"0,43847 (calibrado)","Kaggle 0,45643; holdout 0,44945 inflado"
1,v6,Kneser-Ney modificado; particion agrupada,"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737; holdout 0,41847"
2,v7 ordinal,"Cortes por clase, decodificacion ordinal","0,42321",NO PASA
3,v7 descuento,Interpolacion descontada con concentracion,"0,440875 (theta 0,5) / 0,440788 (theta 1,0)",NO PASA
4,v7 normalizacion,Particion v7_lower; variantes A/B/C,"0,440081 / 0,439709 / 0,436893",NO PASA
5,v7 hiperparametros,8 configuraciones de ordenes y lambda,"mejor C7 0,437677 (dif. media -0,002367)",NO PASA
6,v7 neuronal,GRU de caracteres por clase + KN palabras,(se llena al final),(se llena al final)


## Modelo neuronal y verificaciones de softmax y reproducibilidad

In [5]:
BOS_ID,EOS_ID,UNK_ID=0,1,2

def build_vocab(texts):
    characters=sorted(set(''.join(texts)))
    vocab={'<bos>':BOS_ID,'<eos>':EOS_ID,'<unk>':UNK_ID}
    vocab.update({character:index+3 for index,character in enumerate(characters)})
    return vocab

def encode_chars(texts,vocab):
    return [np.array([vocab.get(character,UNK_ID) for character in text[:NEURAL['max_len']]],dtype=np.int64) for text in texts]

class CharGRU(nn.Module):
    # Entrada one-hot -> GRU(128) -> dropout -> lineal -> softmax sobre el vocabulario. Sin tabla de vectores de entrada.
    def __init__(self,vocab_size):
        super().__init__()
        self.vocab_size=vocab_size
        self.gru=nn.GRU(vocab_size,NEURAL['hidden'],num_layers=1,batch_first=True)
        self.drop=nn.Dropout(NEURAL['dropout'])
        self.out=nn.Linear(NEURAL['hidden'],vocab_size)
    def forward(self,inputs):
        hidden,_=self.gru(F.one_hot(inputs,self.vocab_size).float())
        return self.out(self.drop(hidden))

def make_batch(sequences,indices):
    lengths=np.array([len(sequences[index])+1 for index in indices])
    width=int(lengths.max())
    inputs=np.full((len(indices),width),EOS_ID,dtype=np.int64)
    targets=np.full((len(indices),width),EOS_ID,dtype=np.int64)
    mask=np.zeros((len(indices),width),dtype=bool)
    for row,index in enumerate(indices):
        sequence=sequences[index]
        inputs[row,0]=BOS_ID
        inputs[row,1:len(sequence)+1]=sequence
        targets[row,:len(sequence)]=sequence
        targets[row,len(sequence)]=EOS_ID
        mask[row,:len(sequence)+1]=True
    return torch.from_numpy(inputs),torch.from_numpy(targets),torch.from_numpy(mask)

def training_batches(sequences,rng):
    order=rng.permutation(len(sequences))
    chunk=NEURAL['batch']*32
    batches=[]
    for start in range(0,len(order),chunk):
        block=order[start:start+chunk]
        block=block[np.argsort([len(sequences[index]) for index in block],kind='stable')]
        batches.extend(block[position:position+NEURAL['batch']] for position in range(0,len(block),NEURAL['batch']))
    return [batches[index] for index in rng.permutation(len(batches))]

def sequence_log_likelihood(model,sequences):
    # Log-verosimilitud total de cada texto mas <eos>; log-softmax en float64.
    model.eval()
    totals=np.zeros(len(sequences))
    order=np.argsort([len(sequence) for sequence in sequences],kind='stable')
    with torch.no_grad():
        for start in range(0,len(order),NEURAL['score_batch']):
            indices=order[start:start+NEURAL['score_batch']]
            inputs,targets,mask=make_batch(sequences,indices)
            log_probs=model(inputs).double().log_softmax(dim=-1)
            picked=log_probs.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
            totals[indices]=(picked*mask).sum(dim=1).numpy()
    return totals

def mean_nll_per_char(model,sequences):
    return -sequence_log_likelihood(model,sequences).sum()/sum(len(sequence)+1 for sequence in sequences)

def train_class_model(train_sequences,val_sequences,vocab_size,seed,deadline=None,max_epochs=None,label=''):
    torch.manual_seed(seed)
    rng=np.random.default_rng(seed)
    model=CharGRU(vocab_size)
    assert all(type(module).__name__ not in {'Embedding','EmbeddingBag'} for module in model.modules())
    assert model.gru.input_size==vocab_size,'La GRU debe recibir el one-hot completo.'
    optimizer=torch.optim.Adam(model.parameters(),lr=NEURAL['lr'])
    best_value,best_epoch,best_parameters,bad_epochs,log=math.inf,0,None,0,[]
    for epoch in range(1,(max_epochs or NEURAL['max_epochs'])+1):
        started=time.perf_counter()
        model.train()
        for indices in training_batches(train_sequences,rng):
            inputs,targets,mask=make_batch(train_sequences,indices)
            logits=model(inputs)
            loss=F.cross_entropy(logits[mask],targets[mask])
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),NEURAL['grad_clip'])
            optimizer.step()
        value=mean_nll_per_char(model,val_sequences)
        log.append(dict(modelo=label,epoca=epoch,nll_val_por_caracter=value,segundos=time.perf_counter()-started))
        if value<best_value-1e-4:
            best_value,best_epoch,bad_epochs=value,epoch,0
            best_parameters=[parameter.detach().clone() for parameter in model.parameters()]
        else:
            bad_epochs+=1
        if deadline is not None and time.perf_counter()>deadline:
            raise RuntimeError(f'DETENER: el pliegue supera {FOLD_LIMIT_SECONDS/60:.0f} minutos durante {label}.')
        if bad_epochs>=NEURAL['patience']:
            break
    with torch.no_grad():
        for parameter,best in zip(model.parameters(),best_parameters):
            parameter.copy_(best)
    model.eval()
    return model,dict(modelo=label,mejor_epoca=best_epoch,epocas=len(log),nll_val_mejor=best_value),log

# Verificaciones sobre un subconjunto del ajuste del pliegue 1 (no se usa el pliegue de evaluacion).
probe_texts=texts_development[folds[0][0][:600]]
probe_vocab=build_vocab(probe_texts)
probe_sequences=encode_chars(probe_texts,probe_vocab)
probe_a,_,_=train_class_model(probe_sequences[:500],probe_sequences[500:],len(probe_vocab),seed=SEED,max_epochs=1,label='prueba_a')
probe_b,_,_=train_class_model(probe_sequences[:500],probe_sequences[500:],len(probe_vocab),seed=SEED,max_epochs=1,label='prueba_b')
batch_inputs,batch_targets,batch_mask=make_batch(probe_sequences,np.arange(500,600))
with torch.no_grad():
    probabilities=probe_a(batch_inputs).double().softmax(dim=-1)[batch_mask]
    exp_log_softmax=probe_a(batch_inputs).double().log_softmax(dim=-1)[batch_mask].exp()
sample_rows=np.random.default_rng(SEED).choice(len(probabilities),size=min(500,len(probabilities)),replace=False)
softmax_deviation=float((probabilities[sample_rows].sum(dim=-1)-1).abs().max())
log_softmax_deviation=float((exp_log_softmax[sample_rows].sum(dim=-1)-1).abs().max())
assert softmax_deviation<1e-6 and log_softmax_deviation<1e-6
print('VERIFICACION SOFTMAX:',len(sample_rows),'contextos; max |suma-1| softmax =',softmax_deviation,'; exp(log-softmax usado en la puntuacion) =',log_softmax_deviation,'; tolerancia 1e-6: OK')
scores_a=sequence_log_likelihood(probe_a,probe_sequences[500:])
scores_a_again=sequence_log_likelihood(probe_a,probe_sequences[500:])
scores_b=sequence_log_likelihood(probe_b,probe_sequences[500:])
assert np.array_equal(scores_a,scores_a_again),'La puntuacion cambia al repetir sobre el mismo modelo.'
assert np.array_equal(scores_a,scores_b),'Dos entrenamientos con la misma semilla dan puntuaciones distintas.'
print('VERIFICACION REPRODUCIBILIDAD: mismo lote, dos puntuaciones y dos entrenamientos con semilla',SEED,'-> puntuaciones identicas (igualdad exacta). Ejemplo:',scores_a[:3].round(6).tolist())
print('Parametros por modelo con vocabulario de prueba:',sum(parameter.numel() for parameter in probe_a.parameters()))
del probe_a,probe_b
gc.collect()


VERIFICACION SOFTMAX: 500 contextos; max |suma-1| softmax = 4.440892098500626e-16 ; exp(log-softmax usado en la puntuacion) = 6.661338147750939e-16 ; tolerancia 1e-6: OK
VERIFICACION REPRODUCIBILIDAD: mismo lote, dos puntuaciones y dos entrenamientos con semilla 42 -> puntuaciones identicas (igualdad exacta). Ejemplo: [-350.17186, -476.09889, -241.411977]
Parametros por modelo con vocabulario de prueba: 95577


20

## Etapa 1: pliegue 1
Se continua solo si F1(pliegue 1) - 0,440843 > 0.

In [6]:
fold_results={}
training_logs=[]
model_summaries=[]

def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,labels=np.arange(6),average='macro',zero_division=0)

def run_fold(fold_number):
    fit_idx,app_idx=folds[fold_number-1]
    started=time.perf_counter()
    deadline=started+FOLD_LIMIT_SECONDS
    fit_texts,app_texts=texts_development[fit_idx],texts_development[app_idx]
    fit_labels=y_development[fit_idx]
    assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
    characters=fit_score_orders(fit_texts,fit_labels,app_texts,'char',[REF_CHAR_ORDER],'CV',fold_number)[REF_CHAR_ORDER]
    words=fit_score_orders(fit_texts,fit_labels,app_texts,'word',[KN_WORD_ORDER],'CV',fold_number)[KN_WORD_ORDER]
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    fit_groups=np.unique(groups_development[fit_idx])
    internal_rng=np.random.default_rng(SEED+fold_number)
    val_groups=internal_rng.choice(fit_groups,size=int(round(NEURAL['internal_val_fraction']*len(fit_groups))),replace=False)
    is_val=np.isin(groups_development[fit_idx],val_groups)
    assert not {text.lower() for text in fit_texts[is_val]}&{text.lower() for text in fit_texts[~is_val]}
    vocab=build_vocab(fit_texts)
    fit_sequences=encode_chars(fit_texts,vocab)
    app_sequences=encode_chars(app_texts,vocab)
    unk_rate=float(np.mean(np.concatenate(app_sequences)==UNK_ID))
    neural=np.zeros((len(app_idx),6))
    for class_index in range(6):
        label=f'pliegue{fold_number}_{CLASSES[class_index]}'
        train_sequences=[fit_sequences[index] for index in np.flatnonzero((fit_labels==class_index)&~is_val)]
        val_sequences=[fit_sequences[index] for index in np.flatnonzero((fit_labels==class_index)&is_val)]
        model,summary,log=train_class_model(train_sequences,val_sequences,len(vocab),seed=SEED*1000+fold_number*10+class_index,deadline=deadline,label=label)
        neural[:,class_index]=sequence_log_likelihood(model,app_sequences)
        summary.update(textos_entrenamiento=len(train_sequences),textos_validacion_interna=len(val_sequences))
        model_summaries.append(summary)
        training_logs.extend(log)
        print(label,'mejor epoca',summary['mejor_epoca'],'de',summary['epocas'],'nll val',round(summary['nll_val_mejor'],4),'acumulado s',round(time.perf_counter()-started,1),flush=True)
        del model
        gc.collect()
        if time.perf_counter()>deadline:
            raise RuntimeError(f'DETENER: el pliegue {fold_number} supera {FOLD_LIMIT_SECONDS/60:.0f} minutos.')
    seconds=time.perf_counter()-started
    y_app=y_development[app_idx]
    result=dict(app_idx=app_idx,ref=REF_WEIGHT*characters+(1-REF_WEIGHT)*words+prior,combo=NEURAL_WEIGHT*neural+(1-NEURAL_WEIGHT)*words+prior,neural_only=neural+prior,seconds=seconds,vocab=len(vocab),unk_rate=unk_rate,internal_val_rows=int(is_val.sum()))
    for key in ['ref','combo','neural_only']:
        result['f1_'+key]=macro_f1(y_app,result[key].argmax(axis=1))
    fold_results[fold_number]=result
    print(f'PLIEGUE {fold_number}: {seconds/60:.2f} min; vocabulario {len(vocab)}; tasa <unk> en evaluacion {unk_rate:.2e}; F1 referencia {result["f1_ref"]:.6f}; F1 neuronal+KN {result["f1_combo"]:.6f}; F1 neuronal solo {result["f1_neural_only"]:.6f}',flush=True)
    return result

print('ETAPA 1: solo pliegue 1 de StratifiedKFold(5, shuffle=True, random_state=42) sobre los grupos de desarrollo de v7_lower.')
first=run_fold(1)
assert abs(first['f1_ref']-REF_FOLD1_MEASURED)<5e-6,('Referencia del pliegue 1 recalculada distinta de la medida',first['f1_ref'])
stage1_difference=first['f1_combo']-REF_FOLD1_MEASURED
ETAPA1_PASA=stage1_difference>0
display(pd.DataFrame([dict(pliegue=1,f1_neuronal_kn=first['f1_combo'],f1_referencia_medida=REF_FOLD1_MEASURED,f1_referencia_recalculada=first['f1_ref'],diferencia=stage1_difference,f1_neuronal_solo=first['f1_neural_only'],minutos=first['seconds']/60)]))
display(pd.DataFrame(model_summaries))
print('Diferencia pliegue 1 contra 0.440843:',f'{stage1_difference:+.6f}')
print('CONTINUAR A ETAPA 2' if ETAPA1_PASA else 'NO PASA EN ETAPA 1')


ETAPA 1: solo pliegue 1 de StratifiedKFold(5, shuffle=True, random_state=42) sobre los grupos de desarrollo de v7_lower.


CV 1 char ordenes [4] segundos 7.17


CV 1 word ordenes [3] segundos 6.05


pliegue1_13-17 mejor epoca 8 de 8 nll val 1.8167 acumulado s 44.5


pliegue1_18-24 mejor epoca 8 de 8 nll val 1.8543 acumulado s 75.9


...

pliegue1_65-xx mejor epoca 8 de 8 nll val 1.7072 acumulado s 228.9


PLIEGUE 1: 3.82 min; vocabulario 99; tasa <unk> en evaluacion 4.85e-07; F1 referencia 0.440843; F1 neuronal+KN 0.411131; F1 neuronal solo 0.360561


,pliegue,f1_neuronal_kn,f1_referencia_medida,f1_referencia_recalculada,diferencia,f1_neuronal_solo,minutos
0,1,0.411131,0.440843,0.440843,-0.029712,0.360561,3.815717


,modelo,mejor_epoca,epocas,nll_val_mejor,textos_entrenamiento,textos_validacion_interna
0,pliegue1_13-17,8,8,1.816744,16480,1854
1,pliegue1_18-24,8,8,1.854275,16311,1798
2,pliegue1_25-34,8,8,1.863540,16098,1887
3,pliegue1_35-49,8,8,1.780530,16015,1830
4,pliegue1_50-64,8,8,1.770001,15971,1795
5,pliegue1_65-xx,8,8,1.707232,15164,1722


Diferencia pliegue 1 contra 0.440843: -0.029712
NO PASA EN ETAPA 1


## Etapa 2: cinco pliegues

In [7]:
if ETAPA1_PASA:
    print('ETAPA 2: pliegues 2 a 5 (el pliegue 1 se reutiliza; mismo codigo y semillas).')
    for fold_number in range(2,6):
        run_fold(fold_number)
    oof={key:np.zeros((len(y_development),6)) for key in ['ref','combo','neural_only']}
    for result in fold_results.values():
        for key in oof:
            oof[key][result['app_idx']]=result[key]
    assert sorted(fold_results)==[1,2,3,4,5]
    names={'ref':'REF (KN car 4 + KN pal 3, lambda 0,75)','combo':'NEURONAL + KN pal 3 (0,75/0,25)','neural_only':'NEURONAL solo (descriptivo)'}
    stage2_rows=[]
    for key in ['ref','combo','neural_only']:
        per_fold=np.array([fold_results[number]['f1_'+key] for number in range(1,6)])
        differences=per_fold-np.array([fold_results[number]['f1_ref'] for number in range(1,6)])
        stage2_rows.append(dict(modelo=names[key],f1_macro_oof=macro_f1(y_development,oof[key].argmax(axis=1)),**{f'pliegue_{number}':value for number,value in enumerate(per_fold,1)},diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))))
    stage2=pd.DataFrame(stage2_rows)
    print('TABLA ETAPA 2: F1 macro OOF sin calibrar, F1 por pliegue y diferencia pareada contra la referencia A recalculada (ddof=1)')
    print(stage2.to_string(index=False))
    display(pd.DataFrame([dict(pliegue=number,minutos=fold_results[number]['seconds']/60,vocabulario=fold_results[number]['vocab'],tasa_unk=fold_results[number]['unk_rate'],filas_validacion_interna=fold_results[number]['internal_val_rows']) for number in range(1,6)]))
    display(pd.DataFrame(model_summaries))
else:
    stage2=None
    print('ETAPA 2 OMITIDA: NO PASA EN ETAPA 1.')
print('Tiempos por pliegue (min):',{number:round(result['seconds']/60,2) for number,result in fold_results.items()})


ETAPA 2 OMITIDA: NO PASA EN ETAPA 1.
Tiempos por pliegue (min): {1: 3.82}


## Decision

In [8]:
if ETAPA1_PASA:
    combo_row=stage2.loc[stage2.modelo.str.startswith('NEURONAL +')].iloc[0]
    PASA=bool(combo_row.diferencia_pareada_media>0 and combo_row.diferencia_pareada_media>combo_row.diferencia_pareada_std)
    print('Diferencia pareada media:',f'{combo_row.diferencia_pareada_media:+.6f}','desviacion estandar (ddof=1):',f'{combo_row.diferencia_pareada_std:.6f}')
else:
    PASA=False
print('Regla fijada: media de diferencia pareada >0 y > su desviacion estandar muestral; el neuronal solo no decide.')
print('PASA' if PASA else 'NO PASA')
if not PASA:
    print('NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_lm_neuronal.csv ni modelos_v7_neuronal/.')


Regla fijada: media de diferencia pareada >0 y > su desviacion estandar muestral; el neuronal solo no decide.
NO PASA
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_lm_neuronal.csv ni modelos_v7_neuronal/.


## Holdout (solo si PASA)

In [9]:
def train_full_neural(texts,labels,groups,seed_base,phase):
    unique_groups=np.unique(groups)
    rng=np.random.default_rng(seed_base)
    val_groups=rng.choice(unique_groups,size=int(round(NEURAL['internal_val_fraction']*len(unique_groups))),replace=False)
    is_val=np.isin(groups,val_groups)
    vocab=build_vocab(texts)
    sequences=encode_chars(texts,vocab)
    models=[]
    for class_index in range(6):
        train_sequences=[sequences[index] for index in np.flatnonzero((labels==class_index)&~is_val)]
        val_sequences=[sequences[index] for index in np.flatnonzero((labels==class_index)&is_val)]
        model,summary,_=train_class_model(train_sequences,val_sequences,len(vocab),seed=seed_base*10+class_index,label=f'{phase}_{CLASSES[class_index]}')
        print(phase,CLASSES[class_index],'mejor epoca',summary['mejor_epoca'],'nll val',round(summary['nll_val_mejor'],4),flush=True)
        models.append(model)
    return vocab,models

if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    texts_holdout=train.normalized.iloc[idx_holdout].to_numpy()
    assert not {text.lower() for text in texts_development}&{text.lower() for text in texts_holdout}
    words=fit_score_orders(texts_development,y_development,texts_holdout,'word',[KN_WORD_ORDER],'holdout',0)[KN_WORD_ORDER]
    vocab,models=train_full_neural(texts_development,y_development,groups_development,SEED*1000+90,'holdout')
    holdout_sequences=encode_chars(texts_holdout,vocab)
    neural=np.column_stack([sequence_log_likelihood(model,holdout_sequences) for model in models])
    prior=np.log(np.bincount(y_development,minlength=6)/len(y_development))
    holdout_pred=(NEURAL_WEIGHT*neural+(1-NEURAL_WEIGHT)*words+prior).argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=macro_f1(y_holdout,holdout_pred)
    print('F1 macro holdout v7_lower:',holdout_f1)
    display(pd.DataFrame(classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)).T.loc[CLASSES,['precision','recall','f1-score','support']])
    print('Holdout segundos:',round(time.perf_counter()-started,1),'; evaluacion unica.')
    del models
    gc.collect()
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.


## CSV y pesos (solo si PASA)

In [10]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts=test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    words=fit_score_orders(train.normalized.to_numpy(),y_all,test_texts,'word',[KN_WORD_ORDER],'final',0)[KN_WORD_ORDER]
    vocab,models=train_full_neural(train.normalized.to_numpy(),y_all,group_ids,SEED*1000+99,'final')
    test_sequences=encode_chars(test_texts,vocab)
    neural=np.column_stack([sequence_log_likelihood(model,test_sequences) for model in models])
    prior=np.log(np.bincount(y_all,minlength=6)/len(y_all))
    predictions=(NEURAL_WEIGHT*neural+(1-NEURAL_WEIGHT)*words+prior).argmax(axis=1)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[predictions]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    print('CSV VERIFICADO:',SUB_PATH.name,len(submission),'filas.')
    MODEL_DIR.mkdir()
    with (MODEL_DIR/'.gitignore').open('x',encoding='utf-8') as handle:
        handle.write('*\n!.gitignore\n')
    for class_index,model in enumerate(models):
        torch.save(model.state_dict(),MODEL_DIR/f'gru_clase_{CLASSES[class_index]}.pt')
    with (MODEL_DIR/'vocabulario.json').open('x',encoding='utf-8') as handle:
        json.dump(vocab,handle,ensure_ascii=False)
    for path in sorted(MODEL_DIR.glob('*.pt'))+[MODEL_DIR/'vocabulario.json']:
        assert subprocess.run(['git','check-ignore','-q',str(path)],cwd=OUT_DIR).returncode==0,('No ignorado por git',path.name)
    print('Pesos guardados en',MODEL_DIR.name,'e ignorados por git (git check-ignore).')
else:
    assert not SUB_PATH.exists() and not MODEL_DIR.exists()
    print('CSV OMITIDO: NO PASA; no se carga el test, no se genera salida ni se guardan pesos.')


CSV OMITIDO: NO PASA; no se carga el test, no se genera salida ni se guardan pesos.


## Matriz de confusion del mejor modelo evaluado

In [11]:
if stage2 is not None:
    candidates={'ref':macro_f1(y_development,oof['ref'].argmax(axis=1)),'combo':macro_f1(y_development,oof['combo'].argmax(axis=1))}
    best_key=max(candidates,key=candidates.get)
    y_eval,pred_eval=y_development,oof[best_key].argmax(axis=1)
    scope='OOF de 5 pliegues'
else:
    candidates={'ref':first['f1_ref'],'combo':first['f1_combo']}
    best_key=max(candidates,key=candidates.get)
    y_eval,pred_eval=y_development[first['app_idx']],first[best_key].argmax(axis=1)
    scope='pliegue 1 (unico evaluado)'
label_names={'ref':'referencia A','combo':'neuronal + KN palabras'}
matrix=confusion_matrix(y_eval,pred_eval,labels=np.arange(6))
print('Matriz de confusion del mejor modelo:',label_names[best_key],'|',scope,'| F1 macro',round(candidates[best_key],6))
display(pd.DataFrame(matrix,index=[f'real {name}' for name in CLASSES],columns=[f'pred {name}' for name in CLASSES]))
recall=matrix.diagonal()/matrix.sum(axis=1)
off_diagonal=[(matrix[i,j],CLASSES[i],CLASSES[j]) for i in range(6) for j in range(6) if i!=j]
adjacent_share=sum(matrix[i,j] for i in range(6) for j in range(6) if abs(i-j)==1)/max(1,matrix.sum()-matrix.trace())
print('Recall por clase:',{str(name):round(float(value),3) for name,value in zip(CLASSES,recall)})
print('Mayores confusiones:',[(f'{real}->{pred}',int(count)) for count,real,pred in sorted(off_diagonal,reverse=True)[:4]])
top_recall=[str(CLASSES[index]) for index in np.argsort(-recall)[:2]]
print(f'HIPOTESIS (no verificada): {adjacent_share:.1%} de los errores caen en rangos adyacentes; si es alto, sugiere que el estilo de escritura cambia de forma gradual con la edad y que las fronteras entre rangos vecinos son difusas. Las clases con mayor recall ({", ".join(top_recall)}) podrian tener rasgos de escritura mas marcados. Requiere analisis cualitativo antes de afirmarlo.')


Matriz de confusion del mejor modelo: referencia A | pliegue 1 (unico evaluado) | F1 macro 0.440843


,pred 13-17,pred 18-24,pred 25-34,pred 35-49,pred 50-64,pred 65-xx
real 13-17,2202,1124,478,247,237,170
real 18-24,952,1824,723,381,322,241
real 25-34,512,949,1445,607,538,369
real 35-49,236,481,748,1456,799,603
real 50-64,148,241,393,542,2218,715
real 65-xx,123,241,258,422,745,2372


Recall por clase: {'13-17': 0.494, '18-24': 0.411, '25-34': 0.327, '35-49': 0.337, '50-64': 0.521, '65-xx': 0.57}
Mayores confusiones: [('13-17->18-24', 1124), ('18-24->13-17', 952), ('25-34->18-24', 949), ('35-49->50-64', 799)]
HIPOTESIS (no verificada): 54.3% de los errores caen en rangos adyacentes; si es alto, sugiere que el estilo de escritura cambia de forma gradual con la edad y que las fronteras entre rangos vecinos son difusas. Las clases con mayor recall (65-xx, 50-64) podrian tener rasgos de escritura mas marcados. Requiere analisis cualitativo antes de afirmarlo.


## Resumen

In [12]:
if stage2 is not None:
    combo_oof=float(stage2.loc[stage2.modelo.str.startswith('NEURONAL +'),'f1_macro_oof'].iloc[0])
    ref_oof=float(stage2.loc[stage2.modelo.str.startswith('REF'),'f1_macro_oof'].iloc[0])
    neural_oof=float(stage2.loc[stage2.modelo.str.startswith('NEURONAL solo'),'f1_macro_oof'].iloc[0])
    result_line=f'F1 macro OOF neuronal + KN = {combo_oof:.6f}; referencia A recalculada = {ref_oof:.6f}; neuronal solo (descriptivo) = {neural_oof:.6f}.'
    history.loc[history.version=='v7 neuronal',['f1_oof','estado']]=[f'{combo_oof:.6f}','PASA' if PASA else 'NO PASA']
else:
    result_line=f'Etapa 1: F1 pliegue 1 neuronal + KN = {first["f1_combo"]:.6f} frente a 0,440843 (diferencia {stage1_difference:+.6f}); neuronal solo = {first["f1_neural_only"]:.6f}.'
    history.loc[history.version=='v7 neuronal',['f1_oof','estado']]=[f'pliegue 1: {first["f1_combo"]:.6f}','NO PASA EN ETAPA 1']
display(history)
fold_minutes=', '.join(f'{number}: {result["seconds"]/60:.1f}' for number,result in fold_results.items())
summary_text=f'''## Resumen final: modelo de lenguaje neuronal de caracteres

Resultado: **{'PASA' if PASA else ('NO PASA' if ETAPA1_PASA else 'NO PASA EN ETAPA 1')}**. {result_line} Minutos por pliegue: {fold_minutes}.

{'Holdout evaluado una sola vez: F1 = ' + format(holdout_f1, '.6f') + '; CSV generado con escritura exclusiva y pesos ignorados por git.' if PASA else 'Holdout y CSV omitidos explicitamente por la regla; no se guardaron pesos.'}

El modelo es un modelo de lenguaje propio por clase (GRU de 128 unidades, entrada one-hot de caracteres, entrenado desde cero en cada pliegue) combinado con KN de palabras de orden 3 como en v6; no es un embedding ni usa pesos preentrenados. El resultado neuronal es un resultado de clasificacion generativa por verosimilitud, no de clasificacion discriminativa.

**Optimismo de seleccion:** este modelo se eligio despues de cinco rondas previas dentro de la familia generativa (v7 ordinal, descuento, normalizacion, hiperparametros y la propia v6 como punto de partida), lo que introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6 ni con la particion con fuga de v5; los scores publicos de Kaggle no son comparables con holdout. Referencias historicas: v5 calibrado 0,43847, v6 sin calibrar 0,44106 y v6 calibrado 0,44353, sobre particiones distintas, por lo que la comparacion no es directa.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-neural (torch {torch.__version__} CPU), kernel tweetage-neural.
'''
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


,version,descripcion,f1_oof,estado
0,v5,Kneser-Ney; particion con fuga de 2.561 textos,"0,43847 (calibrado)","Kaggle 0,45643; holdout 0,44945 inflado"
1,v6,Kneser-Ney modificado; particion agrupada,"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737; holdout 0,41847"
2,v7 ordinal,"Cortes por clase, decodificacion ordinal","0,42321",NO PASA
3,v7 descuento,Interpolacion descontada con concentracion,"0,440875 (theta 0,5) / 0,440788 (theta 1,0)",NO PASA
4,v7 normalizacion,Particion v7_lower; variantes A/B/C,"0,440081 / 0,439709 / 0,436893",NO PASA
5,v7 hiperparametros,8 configuraciones de ordenes y lambda,"mejor C7 0,437677 (dif. media -0,002367)",NO PASA
6,v7 neuronal,GRU de caracteres por clase + KN palabras,pliegue 1: 0.411131,NO PASA EN ETAPA 1


## Resumen final: modelo de lenguaje neuronal de caracteres

Resultado: **NO PASA EN ETAPA 1**. Etapa 1: F1 pliegue 1 neuronal + KN = 0.411131 frente a 0,440843 (diferencia -0.029712); neuronal solo = 0.360561. Minutos por pliegue: 1: 3.8.

Holdout y CSV omitidos explicitamente por la regla; no se guardaron pesos.

El modelo es un modelo de lenguaje propio por clase (GRU de 128 unidades, entrada one-hot de caracteres, entrenado desde cero en cada pliegue) combinado con KN de palabras de orden 3 como en v6; no es un embedding ni usa pesos preentrenados. El resultado neuronal es un resultado de clasificacion generativa por verosimilitud, no de clasificacion discriminativa.

**Optimismo de seleccion:** este modelo se eligio despues de cinco rondas previas dentro de la familia generativa (v7 ordinal, descuento, normalizacion, hiperparametros y la propia v6 como punto de partida), lo que introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6 ni con la particion con fuga de v5; los scores publicos de Kaggle no son comparables con holdout. Referencias historicas: v5 calibrado 0,43847, v6 sin calibrar 0,44106 y v6 calibrado 0,44353, sobre particiones distintas, por lo que la comparacion no es directa.

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-neural (torch 2.14.1 CPU), kernel tweetage-neural.


HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
